In [ ]:
!pip install torch torchvision torchaudio
!pip install diffusers
!pip install gradio
!pip install pillow


In [ ]:

import torch
from torch import autocast
from diffusers import StableDiffusionPipeline
import gradio as gr
from PIL import Image

In [ ]:
# Load Stable Diffusion model
authorization_token = ""
model_id = "CompVis/stable-diffusion-v1-4"
device = "cuda"

In [ ]:
pipe = StableDiffusionPipeline.from_pretrained(
    model_id, revision="fp16", torch_dtype=torch.float16, use_auth_token=authorization_token
)
pipe.to(device)

In [ ]:
# generate the image
def generate_image(prompt):
    with autocast("cuda"):
        image = pipe(prompt, guidance_scale=8.5).images[0]
    return image


In [ ]:
import gradio as gr
from diffusers import StableDiffusionPipeline
import torch
from torch import autocast


# Load Stable Diffusion model

authorization_token = "YOUR_HUGGINGFACE_TOKEN"  # <-- put your HF token here
model_id = "CompVis/stable-diffusion-v1-4"
device = "cuda"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    revision="fp16",
    torch_dtype=torch.float16,
    use_auth_token=authorization_token
)
pipe.to(device)

# Function to generate the image

def generate_image(prompt):
    with autocast("cuda"):
        image = pipe(prompt, guidance_scale=8.5).images[0]
    return image


# Theory content

theory_text = """
# 📖 Introduction to Text-to-Image Generation

Text-to-Image Generation is an AI technology that converts **written descriptions** into **visual images**.
This fascinating field uses deep learning models, such as **Stable Diffusion** and **DALL·E**, to interpret
human language and create realistic or artistic images.

### 🌟 Advantages:
1. Generates unique images instantly.
2. No need for manual drawing or design.
3. Boosts creativity for artists and designers.
4. Can produce illustrations for books, games, and movies.
5. Saves time and resources in content creation.

### 🛠 Applications:
- 🎮 **Game Design:** Creating characters, backgrounds, and assets.
- 📚 **Publishing:** Generating book illustrations.
- 📰 **Media:** Producing images for news or blogs.
- 🎨 **Art:** Helping artists visualize ideas.
- 🏭 **Industry:** Product design and prototyping.

Text-to-image generation is revolutionizing creativity by merging **art** and **artificial intelligence**.
It opens a world where imagination can be instantly transformed into visual reality.
"""


# Gradio Interface

def gradio_interface():
    with gr.Blocks(css="""
        .gradio-container {
            background: linear-gradient(to right, #6a11cb, #2575fc);
            color: white;
            font-family: 'Segoe UI', sans-serif;
            min-height: 100vh;
            padding: 20px;
        }
        .main-title {
            font-size: 42px;
            font-weight: bold;
            text-align: center;
            margin-bottom: 20px;
            color: #fff;
        }
        .section-button {
            padding: 16px 40px;
            font-size: 20px;
            font-weight: bold;
            border-radius: 14px;
            border: none;
            cursor: pointer;
            margin: 10px;
        }
        .button-theory {
            background-color: #ff9800;
            color: white;
        }
        .button-theory:hover {
            background-color: #e68900;
        }
        .button-generate {
            background-color: #4caf50;
            color: white;
        }
        .button-generate:hover {
            background-color: #3e8e41;
        }
        .gr-textbox {
            background-color: rgba(255, 255, 255, 0.9);
            border-radius: 10px;
            color: #333;
        }
        .output-image {
            border-radius: 14px;
            box-shadow: 0px 0px 15px rgba(255,255,255,0.6);
        }
    """) as demo:

        gr.Markdown("<div class='main-title'>🎨 Text-Image Generation using fine tuned Stable-diffusion model for context aware visual understanding 📖</div>")

        with gr.Row():
            theory_btn = gr.Button("📚 Theory", elem_classes=["section-button", "button-theory"])
            gen_btn = gr.Button("🖌 Text-to-Image Generation", elem_classes=["section-button", "button-generate"])

        theory_box = gr.Markdown(visible=False)
        prompt_input = gr.Textbox(
            placeholder="✏️ Enter your image description here...",
            label="Enter Prompt",
            visible=False,
            lines=2
        )
        submit_btn = gr.Button("🚀 Generate Image", visible=False)
        output_img = gr.Image(label="🎨 Generated Image", visible=False, elem_classes=["output-image"])

        def show_theory():
            return gr.update(value=theory_text, visible=True), gr.update(visible=False), gr.update(visible=False), gr.update(visible=False)

        def show_generator():
            return gr.update(visible=False), gr.update(visible=True), gr.update(visible=True), gr.update(visible=False)

        def submit_image(prompt):
            img = generate_image(prompt)
            return gr.update(value=img, visible=True)

        theory_btn.click(show_theory, outputs=[theory_box, prompt_input, submit_btn, output_img])
        gen_btn.click(show_generator, outputs=[theory_box, prompt_input, submit_btn, output_img])
        submit_btn.click(submit_image, inputs=[prompt_input], outputs=[output_img])

    return demo

# final
demo = gradio_interface()
demo.launch(debug=True)
